# 5.6 Model Development + 5.7 MLflow Experiment Tracking

This notebook trains at least 5 classifiers (including a baseline), evaluates them, and logs all runs to MLflow.

In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv('../data/clean/validated_data.csv')

price_bins = [0, 5000, 15000, float('inf')]
price_labels = ['budget', 'mid-range', 'luxury']
df['price_tier'] = pd.cut(df['price'], bins=price_bins, labels=price_labels, include_lowest=True).astype(str)

target_col = 'price_tier'
X = df.drop(columns=[target_col, 'price'], errors='ignore')
y = df[target_col].astype(str)

print('Dataset shape:', df.shape)

print('\nClass distribution:')
print((y.value_counts(normalize=True) * 100).round(2))

Dataset shape: (245630, 10)

Class distribution:
price_tier
0    52.90
1    38.45
2     8.65
Name: proportion, dtype: float64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (196504, 8) Test: (49126, 8)


In [4]:
num_cols = X_train.select_dtypes(include=['number']).columns.tolist() 
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist() 

num_pipe = Pipeline([ ('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()) ]) 
cat_pipe = Pipeline([ ('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore')) ])
preprocess = ColumnTransformer([ ('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols) ])

def business_metrics(y_true, y_pred): 
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0) 
    luxury_recall = report.get('luxury', {}).get('recall', 0.0)
    y_t = pd.Series(y_true).reset_index(drop=True)
    y_p = pd.Series(y_pred).reset_index(drop=True)
    severe = ((y_t == 'budget') & (y_p == 'luxury')) | ((y_t == 'luxury') & (y_p == 'budget')) 
    severe_rate = severe.mean()
    return luxury_recall, severe_rate

In [ ]:
# At least 5 models (including baseline) + hyperparameter tuning with CV on TRAIN only
model_configs = {
    'baseline_dummy': {
        'estimator': DummyClassifier(),
        'param_distributions': {'model__strategy': ['most_frequent', 'prior', 'stratified']},
        'n_iter': 3
    },
    'log_reg': {
        'estimator': LogisticRegression(max_iter=2500, class_weight='balanced', solver='saga', random_state=42),
        'param_distributions': {'model__C': np.logspace(-2, 1, 8)},
        'n_iter': 8
    },
    'random_forest': {
        'estimator': RandomForestClassifier(random_state=42, class_weight='balanced_subsample', n_jobs=-1),
        'param_distributions': {
            'model__n_estimators': [200, 300, 500],
            'model__max_depth': [None, 20, 40],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4]
        },
        'n_iter': 10
    },
    'extra_trees': {
        'estimator': ExtraTreesClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
        'param_distributions': {
            'model__n_estimators': [200, 300, 500],
            'model__max_depth': [None, 20, 40],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4]
        },
        'n_iter': 10
    },
    'linear_svc': {
        'estimator': LinearSVC(class_weight='balanced', random_state=42),
        'param_distributions': {
            'model__C': np.logspace(-3, 1, 9),
            'model__loss': ['hinge', 'squared_hinge']
        },
        'n_iter': 8
    }
}

ROOT_MLRUNS = os.path.abspath(os.path.join(os.getcwd(), '..', 'mlruns'))
mlflow.set_tracking_uri(f'file:{ROOT_MLRUNS}')
mlflow.set_experiment('used_car_price_tier_classification')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
best_estimators = {}
best_scores = {}

for model_name, cfg in model_configs.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', cfg['estimator'])])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=cfg['param_distributions'],
        n_iter=cfg['n_iter'],
        scoring={'macro_f1': 'f1_macro', 'balanced_accuracy': 'balanced_accuracy'},
        refit='macro_f1',
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=1,
        return_train_score=False
    )

    with mlflow.start_run(run_name=model_name):
        print(f'\nTraining: {model_name}')
        search.fit(X_train, y_train)

        best_pipe = search.best_estimator_
        best_estimators[model_name] = best_pipe
        best_scores[model_name] = float(search.best_score_)

        best_idx = search.best_index_
        cv_macro_f1_best = float(search.best_score_)
        cv_bal_acc_best = float(search.cv_results_['mean_test_balanced_accuracy'][best_idx])

        cv_runs_df = pd.DataFrame(search.cv_results_['params'])
        cv_runs_df['mean_test_macro_f1'] = search.cv_results_['mean_test_macro_f1']
        cv_runs_df['mean_test_balanced_accuracy'] = search.cv_results_['mean_test_balanced_accuracy']
        cv_runs_df['rank_test_macro_f1'] = search.cv_results_['rank_test_macro_f1']
        cv_runs_df = cv_runs_df.sort_values('rank_test_macro_f1')

        print('Hyperparameter trials for', model_name)
        print(cv_runs_df.to_string(index=False))

        mlflow.log_param('model_name', model_name)
        mlflow.log_param('train_rows', len(X_train))
        mlflow.log_param('test_rows', len(X_test))
        mlflow.log_param('cv_folds', cv.get_n_splits())
        mlflow.log_param('tier_policy', 'budget<=5000,mid-range<=15000,luxury>15000')

        for k, v in search.best_params_.items():
            mlflow.log_param(k, v)

        mlflow.log_metric('cv_macro_f1_best', cv_macro_f1_best)
        mlflow.log_metric('cv_balanced_accuracy_best', cv_bal_acc_best)

        y_oof = cross_val_predict(best_pipe, X_train, y_train, cv=cv, method='predict', n_jobs=-1)
        oof_rep = classification_report(y_train, y_oof, zero_division=0)
        labels = sorted(y_train.unique())
        oof_cm = confusion_matrix(y_train, y_oof, labels=labels)
        oof_cm_df = pd.DataFrame(oof_cm, index=labels, columns=labels)

        with tempfile.TemporaryDirectory() as tmp_dir:
            rep_path = os.path.join(tmp_dir, f'{model_name}_cv_oof_classification_report.txt')
            cm_path = os.path.join(tmp_dir, f'{model_name}_cv_oof_confusion_matrix.csv')
            runs_path = os.path.join(tmp_dir, f'{model_name}_cv_all_runs.csv')

            with open(rep_path, 'w', encoding='utf-8') as f:
                f.write(oof_rep)
            oof_cm_df.to_csv(cm_path, index=True)
            cv_runs_df.to_csv(runs_path, index=False)

            mlflow.log_artifact(rep_path)
            mlflow.log_artifact(cm_path)
            mlflow.log_artifact(runs_path)

        mlflow.sklearn.log_model(best_pipe, artifact_path='model')

        results.append({
            'model': model_name,
            'cv_macro_f1_best': cv_macro_f1_best,
            'cv_balanced_accuracy_best': cv_bal_acc_best
        })

results_df = pd.DataFrame(results).sort_values('cv_macro_f1_best', ascending=False)
results_df

/home/asmaa/.cache/pypoetry/virtualenvs/used-car-value-tier-prediction-th7D_CRA-py3.12/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/27 18:46:59 INFO mlflow.tracking.fluent: Experiment with name 'used_car_price_tier_classification' does not exist. Creating a new experiment.


Fitting 5 folds for each of 3 candidates, totalling 15 fits


2026/04/27 18:47:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 18:47:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fitting 5 folds for each of 8 candidates, totalling 40 fits


/home/asmaa/.cache/pypoetry/virtualenvs/used-car-value-tier-prediction-th7D_CRA-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/asmaa/.cache/pypoetry/virtualenvs/used-car-value-tier-prediction-th7D_CRA-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/asmaa/.cache/pypoetry/virtualenvs/used-car-value-tier-prediction-th7D_CRA-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provide

KeyboardInterrupt: 

In [ ]:
# Select best model by BEST CV macro F1, retrain on full train, evaluate once on test
best_model_name = max(best_scores, key=best_scores.get)
best_model = clone(best_estimators[best_model_name])
best_model.fit(X_train, y_train)

y_test_pred = best_model.predict(X_test)
test_macro_f1 = f1_score(y_test, y_test_pred, average='macro')
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
test_luxury_recall, test_severe_rate = business_metrics(y_test, y_test_pred)

print('Best model selected by CV macro F1:', best_model_name)
print('Best CV macro F1:', best_scores[best_model_name])
print('Test macro_f1:', test_macro_f1)
print('Test balanced_accuracy:', test_bal_acc)
print('Test luxury_recall:', test_luxury_recall)
print('Test severe_misclassification_rate:', test_severe_rate)
print(classification_report(y_test, y_test_pred, zero_division=0))

with mlflow.start_run(run_name=f'final_selected_{best_model_name}_test_eval'):
    mlflow.log_param('selected_model_name', best_model_name)
    mlflow.log_param('selection_metric', 'cv_macro_f1_best')
    mlflow.log_param('train_rows', len(X_train))
    mlflow.log_param('test_rows', len(X_test))
    mlflow.log_metric('selected_model_cv_macro_f1_best', float(best_scores[best_model_name]))
    mlflow.log_metric('test_macro_f1', float(test_macro_f1))
    mlflow.log_metric('test_balanced_accuracy', float(test_bal_acc))
    mlflow.log_metric('test_luxury_recall', float(test_luxury_recall))
    mlflow.log_metric('test_severe_misclassification_rate', float(test_severe_rate))

    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_test_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)

    with tempfile.TemporaryDirectory() as tmp_dir:
        report_path = os.path.join(tmp_dir, f'{best_model_name}_test_classification_report.txt')
        cm_path = os.path.join(tmp_dir, f'{best_model_name}_test_confusion_matrix.csv')
        summary_path = os.path.join(tmp_dir, 'model_cv_summary.csv')

        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(classification_report(y_test, y_test_pred, zero_division=0))
        cm_df.to_csv(cm_path, index=True)
        results_df.to_csv(summary_path, index=False)

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(summary_path)

    mlflow.sklearn.log_model(best_model, artifact_path='model')

## Launch MLflow UI
Run this in terminal from repo root:

`mlflow ui --backend-store-uri ./mlruns --port 5000`

Then open `http://127.0.0.1:5000` and take the runs comparison screenshot for section 5.7.